# The Channel That Lies - Reference Solution

Score every hour by how likely it is that the cheap node's ozone reading is
untrustworthy, using only what the nodes themselves report.

The work is in finding where the signal is. It is not in the ozone channel.

In [ ]:
import numpy as np, pandas as pd, os
from sklearn.ensemble import IsolationForest, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score

SEED = 20260801
PUB = "./dataset/public" if os.path.isdir("./dataset/public") else "./prepared/public"
train = pd.read_csv(f"{PUB}/train.csv", parse_dates=["timestamp"])
test  = pd.read_csv(f"{PUB}/test.csv",  parse_dates=["timestamp"])
feats = [c for c in test.columns if c.startswith(("n1_", "n2_"))]
print(f"train {len(train):,} hours, {train.is_anomalous.mean():.1%} anomalous")
print(f"test  {len(test):,} hours, labels withheld")
print(f"{len(feats)} node channels")

## Validation that matches the split

The real split is chronological, so validation must be too - hold out the tail
of training rather than a random sample. A random KFold would put neighbouring
hours on both sides of the boundary and flatter every model.

In [ ]:
cut = int(len(train) * 0.75)
fit, val = train.iloc[:cut], train.iloc[cut:]
yv = val.is_anomalous.values
print(f"fit {len(fit):,} hours ({fit.is_anomalous.mean():.1%} anomalous)")
print(f"val {len(val):,} hours ({yv.mean():.1%} anomalous)")
print(f"no-skill floor: {yv.mean():.4f}")

## The suspect channel knows nothing about itself

The first thing to try is the node's own ozone reading, and the second is
whether the two nodes disagree.

Watch these two numbers carefully. On this autumn hold-out they look mildly
useful - around 0.39 and 0.37 against a 0.25 floor. On the real winter test
period they collapse to 0.257 and 0.246, which is no better than a constant.
Whatever weak signal the ozone channel carries about its own reliability is a
property of the warm season, and it does not survive.

In [ ]:
print("node1 O3 as score   ", round(average_precision_score(yv, val.n1_O3), 4))
print("node disagreement   ", round(average_precision_score(yv, (val.n1_O3-val.n2_O3).abs()), 4))

## The signal is in the surrounding context

Humidity, temperature and the co-pollutant channels drive the cell's
cross-sensitivity, so they are what carry information about when it misreads.
Hand a model all of them.

In [ ]:
cands = {
    "logistic       ": LogisticRegression(max_iter=5000),
    "random forest  ": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "grad boosting  ": HistGradientBoostingClassifier(random_state=SEED, max_iter=300),
}
for name, m in cands.items():
    p = m.fit(fit[feats], fit.is_anomalous).predict_proba(val[feats])[:, 1]
    print(name, round(average_precision_score(yv, p), 4))

iso = IsolationForest(n_estimators=300, random_state=SEED).fit(fit[feats])
print("isolation forest", round(average_precision_score(yv, -iso.score_samples(val[feats])), 4))
lof = LocalOutlierFactor(novelty=True).fit(fit[feats])
print("local outlier   ", round(average_precision_score(yv, -lof.score_samples(val[feats])), 4))

The tree ensembles win here, and they also win on the real test period - the
local ranking used to define the label keeps the base rate at 25 percent in
both splits, so this hold-out is actually informative about what will happen
after the season turns.

## Refit on all training hours and submit

In [ ]:
final = HistGradientBoostingClassifier(random_state=SEED, max_iter=300)
final.fit(train[feats], train.is_anomalous)

os.makedirs("./working", exist_ok=True)
sub = pd.DataFrame({"timestamp": test.timestamp,
                    "score": final.predict_proba(test[feats])[:, 1]})
sub.to_csv("./working/submission.csv", index=False)
print(f"wrote {len(sub):,} rows")
sub.head(3)

## On the real test set

This solver scores **0.552** average precision against a no-skill floor of
0.258. Scoring by the node's own ozone channel gets 0.257 - no better than a
constant.

The gap to 1.0 is not slack. A sensor at this price does not carry enough
information to know when it is wrong, so many of the anomalous hours look,
from the node's own output, exactly like ordinary ones.